# Hello World: From Raw CSV to Governed Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb)

## Business Scenario

You have a raw CSV of employee data and need to:

1. **Enforce a schema** — validate column types and required fields
2. **Apply quality rules** — catch invalid emails, out-of-range salaries, unknown departments
3. **Add derived columns** — enrich with computed fields like tenure category
4. **Materialize output** — write validated data to Parquet
5. **Inspect the run report** — see exactly what LakeLogic tracked

All of this with **one contract** and **three lines of code**.

---

## What You'll Learn

| Section | Concept |
|---|---|
| 1 | In-memory contract — schema + quality + transformations |
| 2 | YAML file contract — production-ready, Git-versioned |
| 3 | Materialization — write validated data to Parquet |
| 4 | Run Report — what LakeLogic tracked automatically |
| 5 | From notebook to production — next steps |


## Setup

In [ ]:
import subprocess, sys, os, urllib.request
from pathlib import Path

# Install/upgrade lakelogic with polars engine (v1.5+ required)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "-q",
     "lakelogic[polars]>=1.5.0"],
    check=True,
)

import lakelogic
print(f"lakelogic v{lakelogic.__version__} ready.")

# On Colab: clone repo for contract files and sample data
if "google.colab" in sys.modules:
    repo = Path("/content/LakeLogic")
    if repo.exists():
        import shutil
        shutil.rmtree(repo)
    subprocess.run(
        ["git", "clone", "--quiet",
         "https://github.com/lakelogic/LakeLogic.git", str(repo)],
        check=True,
    )
    os.chdir(repo / "examples" / "01_quickstart")
    print(f"Working directory: {Path.cwd()}")

from lakelogic import DataProcessor

REMOTE_URL = (
    "https://raw.githubusercontent.com/lakelogic/LakeLogic/main/examples/01_quickstart/files/excel/data/employees.csv"
)
print(f"Source URL: {REMOTE_URL}")
print("Setup complete.")

## Engine Selection

LakeLogic supports multiple engines from the same contract. Choose your engine below.

> **Spark note:** Spark cannot read files directly from HTTPS URLs (the Hadoop `HttpsFileSystem`
> driver doesn't implement `listStatus`). When `engine = 'spark'`, the cell below automatically
> downloads the file to `/tmp/` first and passes the local path to LakeLogic.
> All other engines (Polars, DuckDB, Pandas) read the remote URL directly.


In [ ]:
# ── Choose your engine ──────────────────────────────────────────────────
# Options: 'polars' | 'duckdb' | 'pandas' | 'spark'
ENGINE = "polars"  # change this to try a different engine

# ── Resolve source path ──────────────────────────────────────────────────
# Spark can't read from HTTPS URLs directly — download locally first.
if ENGINE == "spark":
    LOCAL_FILE = "/tmp/lakelogic_employees.csv"
    if not os.path.exists(LOCAL_FILE):
        print(f"Downloading {REMOTE_URL} for Spark...")
        urllib.request.urlretrieve(REMOTE_URL, LOCAL_FILE)
    SOURCE = LOCAL_FILE
    print(f"Spark engine: using local file → {SOURCE}")
else:
    SOURCE = REMOTE_URL
    print(f"Engine: {ENGINE} | Source: {SOURCE}")

---

## 1. In-Memory Contract: Schema + Quality + Transformations

Define everything in a Python dict — great for prototyping. This contract:

- **Schema:** Enforces column types and required fields
- **Quality rules:** Validates email format, salary range, department values, and active status
- **Transformations:** Derives `department_upper` and `tenure_category` columns


In [ ]:
contract_dict = {
    "version": "1.0.0",
    "dataset": "employees",
    "source": {"type": "landing"},
    
    # ── Schema: define expected columns and types ──────────────────────
    "model": {
        "fields": [
            {"name": "id",         "type": "integer", "required": True},
            {"name": "name",       "type": "string",  "required": True},
            {"name": "email",      "type": "string",  "required": True},
            {"name": "department", "type": "string"},
            {"name": "salary",     "type": "integer"},
            {"name": "hire_date",  "type": "string"},
            {"name": "status",     "type": "string"},
        ]
    },

    # ── Transformations: enrich the data ──────────────────────────────
    "transformations": [
        {
            "phase": "pre",
            "derive": {
                "field": "department_upper",
                "sql": "UPPER(department)",
            },
        },
        {
            "phase": "pre",
            "derive": {
                "field": "tenure_category",
                "sql": "CASE WHEN hire_date >= '2024-01-01' THEN 'New Hire' "
                       "WHEN hire_date >= '2023-01-01' THEN 'Mid-Tenure' "
                       "ELSE 'Veteran' END",
            },
        },
    ],

    # ── Quality rules: validate every row ─────────────────────────────
    "quality": {
        "row_rules": [
            {"name": "valid_email",      "sql": "email LIKE '%@%'",        "category": "validity"},
            {"name": "has_name",         "sql": "name IS NOT NULL AND name != ''", "category": "completeness"},
            {"name": "valid_salary",     "sql": "salary > 0 AND salary < 500000",  "category": "validity"},
            {"name": "valid_department", "sql": "department IN ('Engineering', 'Sales', 'HR', 'Marketing', 'Finance')", "category": "validity"},
            {"name": "is_active",        "sql": "status = 'active'",       "category": "validity"},
        ]
    },
}

print("Contract defined with:")
print(f"  Schema fields:    {len(contract_dict['model']['fields'])}")
print(f"  Transformations:  {len(contract_dict['transformations'])}")
print(f"  Quality rules:    {len(contract_dict['quality']['row_rules'])}")

In [ ]:
# ── Run the contract ─────────────────────────────────────────────────
processor = DataProcessor(contract=contract_dict, engine=ENGINE)
result = processor.run_source(SOURCE)

print(f"Engine      : {ENGINE}")
print(f"Source rows : {result.source_count}")
print(f"Good rows   : {result.good_count}")
print(f"Bad rows    : {result.bad_count}")

In [ ]:
# ── Inspect results ──────────────────────────────────────────────────
def show_df(label, df):
    """Display a DataFrame — handles Spark vs Polars/Pandas/DuckDB."""
    print(f"\n{label}")
    print("─" * 60)
    if hasattr(df, "toPandas"):  # Spark DataFrame
        df.show(20, truncate=False)
    else:
        try:
            display(df)  # Jupyter display for Polars/Pandas
        except Exception:
            print(df)

show_df("✅ GOOD rows (passed all rules):", result.good)
show_df("❌ BAD rows (quarantined):", result.bad)

# Notice the derived columns: department_upper, tenure_category
print("\n📋 Output columns (including derived):")
if hasattr(result.good, 'columns'):
    print(f"  {list(result.good.columns)}")
elif hasattr(result.good, 'schema'):
    print(f"  {result.good.schema.names}")

---

## 2. YAML File Contract — Production-Ready

In production, contracts live in `.yaml` files tracked in Git. The code stays the same;
the contract controls everything.

Here's what our production contract looks like:

```yaml
version: 1.0.0
dataset: employees

info:
  title: Employee Directory (Quickstart)
  version: 1.0.0
  description: >
    Validates and cleanses employee CSV data.

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    # ... (see employees_contract.yaml)

transformations:
  - phase: pre
    derive:
      field: tenure_category
      sql: "CASE WHEN hire_date >= '2024-01-01' THEN 'New Hire' ... END"

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%'"

materialization:
  target_path: ./output/employees_validated
  format: parquet
  strategy: overwrite
```

> **Key difference:** The YAML contract includes `materialization` — it will write the validated
> data to Parquet automatically. The code is identical.


In [ ]:
# ── Same code, YAML contract ─────────────────────────────────────────
processor_yaml = DataProcessor(contract="employees_contract.yaml", engine=ENGINE)
result_yaml = processor_yaml.run_source(SOURCE)

print(f"YAML contract results:")
print(f"  Good: {result_yaml.good_count}  |  Bad: {result_yaml.bad_count}")

show_df("✅ GOOD rows:", result_yaml.good)
show_df("❌ BAD rows:", result_yaml.bad)

---

## 3. Materialize — Write Validated Data to Parquet

The YAML contract includes `materialization.target_path`, so LakeLogic can write
the validated `good` rows directly to Parquet. No extra code needed.


In [ ]:
# ── Materialize the validated data ────────────────────────────────────
mat_result = processor_yaml.materialize(result_yaml.good, result_yaml.bad)

print(f"Materialization result: {mat_result}")

# ── Verify the output ────────────────────────────────────────────────
output_path = Path("./output/employees_validated")
if output_path.exists():
    output_files = list(output_path.rglob("*"))
    print(f"\n📁 Output directory: {output_path}")
    print(f"   Files written: {len([f for f in output_files if f.is_file()])}")
    for f in output_files:
        if f.is_file():
            print(f"   └── {f.name} ({f.stat().st_size:,} bytes)")

---

## 4. Run Report — What LakeLogic Tracked

Every run produces a detailed report with row counts, timing, rule results,
and lineage metadata — automatically.


In [ ]:
# ── Inspect the run report ────────────────────────────────────────────
report = processor_yaml.last_report

if report:
    print("📊 Run Report")
    print("═" * 60)
    print(f"  Contract      : {report.get('contract', '?')}")
    print(f"  Run ID        : {report.get('run_id', '?')}")
    print(f"  Timestamp     : {report.get('timestamp', '?')}")
    print(f"  Engine        : {report.get('engine', '?')}")
    print(f"  Dataset       : {report.get('dataset', '?')}")
    print(f"  Domain        : {report.get('domain', '?')}")
    print(f"  Data Layer    : {report.get('data_layer', '?')}")
    print()
    
    # Row counts
    counts = report.get('counts', {})
    print("  📈 Row Counts:")
    print(f"    Source       : {counts.get('source', '?')}")
    print(f"    Total (good) : {counts.get('total', '?')}")
    print(f"    Quarantined  : {counts.get('quarantined', '?')}")
    print()
    
    # Quality rule failures
    failures = report.get('row_rule_failures', [])
    if failures:
        print("  ❌ Quality Rule Failures:")
        for r in failures:
            if isinstance(r, dict):
                print(f"    • {r.get('rule', r.get('name', '?'))}: {r.get('failed_count', r.get('count', '?'))} rows")
            else:
                print(f"    • {r}")
    else:
        print("  ✅ No quality rule failures in report!")
    print()
    
    # Schema drift
    drift = report.get('schema_drift', {})
    if drift:
        print(f"  🔄 Schema Drift: {drift}")
    
    print("═" * 60)
    
    # Full report for exploration
    print("\n🔍 Full report keys:", list(report.keys()))
else:
    print("No report available — run the processor first.")

---

## 5. Summary

### What LakeLogic Did Automatically

| Step | What happened |
|---|---|
| **Schema** | Validated column types and required fields |
| **Transformations** | Derived `department_upper` and `tenure_category` |
| **Quality** | Applied 5 rules, split data into good / bad |
| **Materialization** | Wrote validated rows to Parquet |
| **Audit** | Added `_lakelogic_processed_at` and `_lakelogic_run_id` columns |
| **Run Report** | Captured row counts, timing, rule results, drift |

### Dict vs YAML — When to Use Each

| Format | Best for | Key difference |
|---|---|---|
| Python `dict` | Prototyping, dynamic generation, notebooks | Quick iteration, inline with code |
| YAML file | Production, Git versioning, team governance | Declarative, reviewable, version-controlled |

Both produce identical results — switch between them with zero code change.

---

### From Notebook to Production

This quickstart showed a single contract processing one file. In production, LakeLogic scales to:

| Capability | How |
|---|---|
| **Multi-layer pipelines** | Chain bronze → silver → gold with `_system.yaml` |
| **DAG visualization** | See contract dependencies as an interactive graph |
| **Incremental processing** | Only process new data with `load_mode: incremental` |
| **Schema evolution** | Control per-layer with `server_defaults` |
| **Reprocessing** | Fix historical data without rebuilding the full pipeline |
| **GDPR compliance** | Soft-delete or hash PII with `forget_*` parameters |

### Next Quickstarts

- **[`02_pipeline_quickstart`](../02_pipeline_quickstart/02_pipeline_quickstart.ipynb)** — Build a bronze → silver → gold pipeline with DAG visualization
- **[`02_database_governance.ipynb`](02_database_governance.ipynb)** — Quality-gate pattern applied to a SQLite database
- **[`SCD2 dimension`](../02_core_patterns/scd2_dimension/playbook.ipynb)** — SCD2 history tracking
- **[`Soft delete`](../02_core_patterns/soft_delete/soft_delete_pattern.ipynb)** — Flag deletes instead of removing rows
